# Inter-Annotator Agreement untuk Dataset Sentimen Gojek

Notebook ini menghitung tingkat kesepakatan antar anotator menggunakan:
- **Fleiss' Kappa** — kesepakatan semua 3 anotator sekaligus
- **Cohen's Kappa** — kesepakatan berpasangan (1 vs 2, 1 vs 3, 2 vs 3)
- **Krippendorff's Alpha** — ukuran yang lebih robust, cocok untuk data nominal
- **Percentage Agreement** — persentase baris yang labelnya sama persis

Sebelum menghitung kappa, kita verifikasi dulu bahwa kolom `sentence_id`, `at`, dan `content` **persis sama** di ketiga annotator.

## Langkah 1 — Install & Import Library

In [19]:
import sys, subprocess

# Install library yang dibutuhkan untuk kappa calculation
# - krippendorff : untuk Krippendorff's Alpha
# - statsmodels  : untuk Fleiss' Kappa
pkgs = ["krippendorff", "statsmodels"]
for pkg in pkgs:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✅ {pkg} siap")
    else:
        print(f"❌ {pkg} gagal: {result.stderr[-200:]}")

✅ krippendorff siap
✅ statsmodels siap


In [20]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score          # Cohen's Kappa (pairwise)
import krippendorff                                    # Krippendorff's Alpha
from statsmodels.stats.inter_rater import (
    fleiss_kappa, aggregate_raters                     # Fleiss' Kappa
)
import warnings
warnings.filterwarnings("ignore")

print("✅ Semua library berhasil diimport")

✅ Semua library berhasil diimport


## Langkah 2 — Load Dataset dari `very_clean_dataset/`

Kita load ketiga file CSV annotator dari folder `very_clean_dataset/`.  
Setiap file memiliki kolom: `sentence_id`, `at`, `content`, `category`, `sentiment`.

In [21]:
import os

BASE = r"c:\Users\FATISDA UNS\Documents\Reni\preprocessing\very_clean_dataset"

df1 = pd.read_csv(os.path.join(BASE, "annotator1_clean.csv"), dtype=str).fillna("")
df2 = pd.read_csv(os.path.join(BASE, "annotator2_clean.csv"), dtype=str).fillna("")
df3 = pd.read_csv(os.path.join(BASE, "annotator3_clean.csv"), dtype=str).fillna("")

print(f"Annotator 1 : {len(df1)} baris | kolom: {list(df1.columns)}")
print(f"Annotator 2 : {len(df2)} baris | kolom: {list(df2.columns)}")
print(f"Annotator 3 : {len(df3)} baris | kolom: {list(df3.columns)}")

# Preview sekilas
df1.head(3)

Annotator 1 : 38345 baris | kolom: ['sentence_id', 'at', 'content', 'category', 'sentiment']
Annotator 2 : 38345 baris | kolom: ['sentence_id', 'at', 'content', 'sentiment', 'category']
Annotator 3 : 38345 baris | kolom: ['sentence_id', 'at', 'content', 'category', 'sentiment']


,sentence_id,at,content,category,sentiment
0,review_1_1,2025-10-30 23:58:11,"Map suka eror,, driver dimana map dimana..biki...",app,negative
1,review_2_1,2025-10-30 23:42:51,terlalu mahal skrg.,service,negative
2,review_3_1,2025-10-30 23:34:57,bolehh lah,none,positive


## Langkah 3 — Cek Konsistensi Data (sentence_id, at, content)

Sebelum menghitung agreement, pastikan ketiga annotator bekerja pada **baris yang sama persis**:
- `sentence_id` — ID review harus identik
- `at` — timestamp review harus identik  
- `content` — teks review harus identik

Kalau ada perbedaan, hasil kappa tidak valid karena annotator membandingkan data berbeda.

In [22]:
CHECK_COLS = ["sentence_id", "at", "content"]

def cek_konsistensi(dfA, dfB, nama_A, nama_B):
    """
    Membandingkan kolom CHECK_COLS antara dua dataframe.
    Mengembalikan True jika semua baris identik.
    """
    if len(dfA) != len(dfB):
        print(f"  ⚠️  Jumlah baris berbeda: {nama_A}={len(dfA)}, {nama_B}={len(dfB)}")
        return False

    mismatch = []
    for col in CHECK_COLS:
        diff_mask = dfA[col].values != dfB[col].values
        n_diff = diff_mask.sum()
        if n_diff > 0:
            mismatch.append((col, n_diff))
            # Tampilkan contoh perbedaan (maksimal 3 baris pertama)
            idx_diff = np.where(diff_mask)[0][:3]
            print(f"\n  ⚠️  Kolom '{col}' — {n_diff} baris berbeda antara {nama_A} vs {nama_B}:")
            for i in idx_diff:
                print(f"    Baris {i}: [{nama_A}] '{dfA[col].iloc[i]}'  vs  [{nama_B}] '{dfB[col].iloc[i]}'")

    if not mismatch:
        print(f"  ✅ {nama_A} vs {nama_B} : sentence_id, at, content — IDENTIK semua ({len(dfA)} baris)")
        return True
    else:
        return False

print("=" * 60)
print("CEK KONSISTENSI DATA ANTAR ANNOTATOR")
print("=" * 60)

ok_12 = cek_konsistensi(df1, df2, "Annotator1", "Annotator2")
ok_13 = cek_konsistensi(df1, df3, "Annotator1", "Annotator3")
ok_23 = cek_konsistensi(df2, df3, "Annotator2", "Annotator3")

print("\n" + "=" * 60)
if ok_12 and ok_13 and ok_23:
    print("✅ SEMUA DATA KONSISTEN — aman lanjut ke perhitungan agreement!")
else:
    print("❌ ADA KETIDAKKONSISTENAN — perlu perbaikan data sebelum lanjut!")
print("=" * 60)

CEK KONSISTENSI DATA ANTAR ANNOTATOR
  ✅ Annotator1 vs Annotator2 : sentence_id, at, content — IDENTIK semua (38345 baris)
  ✅ Annotator1 vs Annotator3 : sentence_id, at, content — IDENTIK semua (38345 baris)
  ✅ Annotator2 vs Annotator3 : sentence_id, at, content — IDENTIK semua (38345 baris)

✅ SEMUA DATA KONSISTEN — aman lanjut ke perhitungan agreement!


## Langkah 4 — Fleiss' Kappa

**Fleiss' Kappa** mengukur agreement untuk **lebih dari 2 annotator** sekaligus.

### Interpretasi nilai Kappa:
| Nilai        | Interpretasi         |
|-------------|----------------------|
| < 0.00      | Poor agreement       |
| 0.00 – 0.20 | Slight agreement     |
| 0.21 – 0.40 | Fair agreement       |
| 0.41 – 0.60 | Moderate agreement   |
| 0.61 – 0.80 | Substantial agreement|
| 0.81 – 1.00 | Almost perfect agreement |

Formula Fleiss' Kappa:
$$\kappa = \frac{\bar{P} - \bar{P_e}}{1 - \bar{P_e}}$$

In [23]:
def hitung_fleiss_kappa(col):
    """
    Menghitung Fleiss' Kappa untuk kolom tertentu (category / sentiment).
    
    aggregate_raters() dari statsmodels mengubah data menjadi
    matriks (n_items x n_categories) — jumlah annotator yang memilih tiap label.
    Kemudian fleiss_kappa() menghitung kappanya.
    """
    # Ambil label dari ketiga annotator untuk kolom ini
    ratings = np.column_stack([
        df1[col].values,
        df2[col].values,
        df3[col].values
    ])

    # aggregate_raters: (data, n_cat=None) → (table, categories)
    # table shape: (n_items, n_categories)
    table, _ = aggregate_raters(ratings)

    kappa = fleiss_kappa(table, method="fleiss")
    return kappa

print("=" * 55)
print("FLEISS' KAPPA (3 Annotator Sekaligus)")
print("=" * 55)

fk_cat  = hitung_fleiss_kappa("category")
fk_sent = hitung_fleiss_kappa("sentiment")

print(f"  📌 Category  : {fk_cat:.4f}")
print(f"  📌 Sentiment : {fk_sent:.4f}")

FLEISS' KAPPA (3 Annotator Sekaligus)
  📌 Category  : 0.5354
  📌 Sentiment : 0.6735


## Langkah 5 — Cohen's Kappa (Pairwise)

**Cohen's Kappa** mengukur agreement **antara dua annotator**.  
Kita hitung untuk 3 pasangan: Ann1-Ann2, Ann1-Ann3, Ann2-Ann3.

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

- $p_o$ = observed agreement (proporsi baris yang sama labelnya)
- $p_e$ = expected agreement by chance (baseline acak)

In [24]:
def hitung_cohen_kappa_all(col):
    """
    Menghitung Cohen's Kappa untuk semua pasangan annotator
    pada kolom yang diberikan (category / sentiment).
    """
    pairs = [
        (df1, df2, "Ann1 vs Ann2"),
        (df1, df3, "Ann1 vs Ann3"),
        (df2, df3, "Ann2 vs Ann3"),
    ]
    results = {}
    for dfA, dfB, label in pairs:
        # Hanya hitung baris di mana kedua annotator punya label (tidak kosong)
        mask = (dfA[col] != "") & (dfB[col] != "")
        k = cohen_kappa_score(dfA.loc[mask, col], dfB.loc[mask, col])
        results[label] = k
    return results

print("=" * 55)
print("COHEN'S KAPPA (Pairwise)")
print("=" * 55)

ck_cat  = hitung_cohen_kappa_all("category")
ck_sent = hitung_cohen_kappa_all("sentiment")

print("\n📂 Category:")
for pasangan, k in ck_cat.items():
    print(f"   {pasangan} : {k:.4f}")

print("\n💬 Sentiment:")
for pasangan, k in ck_sent.items():
    print(f"   {pasangan} : {k:.4f}")

COHEN'S KAPPA (Pairwise)

📂 Category:
   Ann1 vs Ann2 : 0.5236
   Ann1 vs Ann3 : 0.5427
   Ann2 vs Ann3 : 0.5492

💬 Sentiment:
   Ann1 vs Ann2 : 0.6635
   Ann1 vs Ann3 : 0.6010
   Ann2 vs Ann3 : 0.7587


## Langkah 6 — Krippendorff's Alpha

**Krippendorff's Alpha** adalah metrik yang lebih general:
- Bisa untuk lebih dari 2 annotator
- Bisa handle **missing data**
- Mendukung berbagai skala: nominal, ordinal, interval, ratio

Untuk label kategorikal (category & sentiment), kita gunakan `level_of_measurement="nominal"`.

$$\alpha = 1 - \frac{D_o}{D_e}$$

- $D_o$ = observed disagreement
- $D_e$ = expected disagreement by chance

In [25]:
def hitung_krippendorff(col):
    """
    Menghitung Krippendorff's Alpha untuk kolom tertentu.
    
    krippendorff.alpha() menerima reliability_data berbentuk:
        np.array shape (n_annotators, n_items)
    Label string diubah ke angka dulu.
    Sel kosong ("") diisi NaN (dianggap missing).
    """
    # Kumpulkan semua label unik (dari ketiga annotator)
    all_labels = sorted(set(df1[col]) | set(df2[col]) | set(df3[col]) - {""})
    label2id = {lbl: i for i, lbl in enumerate(all_labels)}

    def encode(series):
        return series.map(lambda x: label2id[x] if x != "" else np.nan)

    data = np.array([
        encode(df1[col]),
        encode(df2[col]),
        encode(df3[col]),
    ], dtype=float)

    alpha = krippendorff.alpha(
        reliability_data=data,
        level_of_measurement="nominal"
    )
    return alpha

print("=" * 55)
print("KRIPPENDORFF'S ALPHA")
print("=" * 55)

ka_cat  = hitung_krippendorff("category")
ka_sent = hitung_krippendorff("sentiment")

print(f"  📌 Category  : {ka_cat:.4f}")
print(f"  📌 Sentiment : {ka_sent:.4f}")

KRIPPENDORFF'S ALPHA
  📌 Category  : 0.5354
  📌 Sentiment : 0.6735


## Langkah 7 — Percentage Agreement

**Percentage Agreement** adalah cara paling sederhana mengukur agreement:  
persentase item di mana annotator memberikan label yang **sama persis**.

Tidak memperhitungkan agreement by chance, tapi mudah diinterpretasi.

$$\text{PA} = \frac{\text{jumlah baris sama}}{\text{total baris}} \times 100\%$$

In [26]:
def hitung_pa(col):
    """
    Menghitung Percentage Agreement:
    - Per pasangan annotator (pairwise)
    - Semua 3 annotator setuju (all-3 agreement)
    """
    n = len(df1)
    results = {}

    # Pairwise
    for dfA, dfB, label in [
        (df1, df2, "Ann1 vs Ann2"),
        (df1, df3, "Ann1 vs Ann3"),
        (df2, df3, "Ann2 vs Ann3"),
    ]:
        agree = (dfA[col].values == dfB[col].values).sum()
        results[label] = (agree / n) * 100

    # All-3: ketiga annotator setuju persis
    all3 = (
        (df1[col].values == df2[col].values) &
        (df2[col].values == df3[col].values)
    ).sum()
    results["Semua 3 Setuju"] = (all3 / n) * 100

    return results

print("=" * 55)
print("PERCENTAGE AGREEMENT (%)")
print("=" * 55)

pa_cat  = hitung_pa("category")
pa_sent = hitung_pa("sentiment")

print("\n📂 Category:")
for label, pct in pa_cat.items():
    print(f"   {label} : {pct:.2f}%")

print("\n💬 Sentiment:")
for label, pct in pa_sent.items():
    print(f"   {label} : {pct:.2f}%")

PERCENTAGE AGREEMENT (%)

📂 Category:
   Ann1 vs Ann2 : 67.90%
   Ann1 vs Ann3 : 69.13%
   Ann2 vs Ann3 : 70.47%
   Semua 3 Setuju : 56.17%

💬 Sentiment:
   Ann1 vs Ann2 : 81.18%
   Ann1 vs Ann3 : 76.89%
   Ann2 vs Ann3 : 85.46%
   Semua 3 Setuju : 72.62%


## Langkah 8 — Ringkasan Semua Metrik Agreement

Tampilkan semua hasil dalam satu tabel yang rapi untuk mudah dibandingkan.

In [27]:
def interpret_kappa(k):
    """Memberikan label interpretasi berdasarkan nilai kappa."""
    if k < 0:
        return "Poor"
    elif k < 0.20:
        return "Slight"
    elif k < 0.40:
        return "Fair"
    elif k < 0.60:
        return "Moderate"
    elif k < 0.80:
        return "Substantial"
    else:
        return "Almost Perfect"

# ---- Susun baris-baris tabel ----
rows = []

# Fleiss' Kappa
rows.append({"Metrik": "Fleiss' Kappa",     "Pasangan": "Semua 3 Annotator", "Category":  f"{fk_cat:.4f}  ({interpret_kappa(fk_cat)})",  "Sentiment": f"{fk_sent:.4f}  ({interpret_kappa(fk_sent)})"})

# Cohen's Kappa
for pasangan in ["Ann1 vs Ann2", "Ann1 vs Ann3", "Ann2 vs Ann3"]:
    rows.append({
        "Metrik": "Cohen's Kappa",
        "Pasangan": pasangan,
        "Category":  f"{ck_cat[pasangan]:.4f}  ({interpret_kappa(ck_cat[pasangan])})",
        "Sentiment": f"{ck_sent[pasangan]:.4f}  ({interpret_kappa(ck_sent[pasangan])})",
    })

# Krippendorff's Alpha
rows.append({"Metrik": "Krippendorff's Alpha", "Pasangan": "Semua 3 Annotator", "Category":  f"{ka_cat:.4f}  ({interpret_kappa(ka_cat)})",  "Sentiment": f"{ka_sent:.4f}  ({interpret_kappa(ka_sent)})"})

# Percentage Agreement
for pasangan in ["Ann1 vs Ann2", "Ann1 vs Ann3", "Ann2 vs Ann3", "Semua 3 Setuju"]:
    rows.append({
        "Metrik": "Percentage Agreement",
        "Pasangan": pasangan,
        "Category":  f"{pa_cat[pasangan]:.2f}%",
        "Sentiment": f"{pa_sent[pasangan]:.2f}%",
    })

df_summary = pd.DataFrame(rows, columns=["Metrik", "Pasangan", "Category", "Sentiment"])

print("\n" + "=" * 80)
print("RINGKASAN INTER-ANNOTATOR AGREEMENT")
print("=" * 80)
print(df_summary.to_string(index=False))
print("=" * 80)

df_summary


RINGKASAN INTER-ANNOTATOR AGREEMENT
              Metrik          Pasangan           Category             Sentiment
       Fleiss' Kappa Semua 3 Annotator 0.5354  (Moderate) 0.6735  (Substantial)
       Cohen's Kappa      Ann1 vs Ann2 0.5236  (Moderate) 0.6635  (Substantial)
       Cohen's Kappa      Ann1 vs Ann3 0.5427  (Moderate) 0.6010  (Substantial)
       Cohen's Kappa      Ann2 vs Ann3 0.5492  (Moderate) 0.7587  (Substantial)
Krippendorff's Alpha Semua 3 Annotator 0.5354  (Moderate) 0.6735  (Substantial)
Percentage Agreement      Ann1 vs Ann2             67.90%                81.18%
Percentage Agreement      Ann1 vs Ann3             69.13%                76.89%
Percentage Agreement      Ann2 vs Ann3             70.47%                85.46%
Percentage Agreement    Semua 3 Setuju             56.17%                72.62%


,Metrik,Pasangan,Category,Sentiment
0,Fleiss' Kappa,Semua 3 Annotator,0.5354 (Moderate),0.6735 (Substantial)
1,Cohen's Kappa,Ann1 vs Ann2,0.5236 (Moderate),0.6635 (Substantial)
2,Cohen's Kappa,Ann1 vs Ann3,0.5427 (Moderate),0.6010 (Substantial)
3,Cohen's Kappa,Ann2 vs Ann3,0.5492 (Moderate),0.7587 (Substantial)
4,Krippendorff's Alpha,Semua 3 Annotator,0.5354 (Moderate),0.6735 (Substantial)
5,Percentage Agreement,Ann1 vs Ann2,67.90%,81.18%
6,Percentage Agreement,Ann1 vs Ann3,69.13%,76.89%
7,Percentage Agreement,Ann2 vs Ann3,70.47%,85.46%
8,Percentage Agreement,Semua 3 Setuju,56.17%,72.62%
